# LC 435 — Non-overlapping Intervals
**Difficulty:** Medium &nbsp;|&nbsp; **Category:** Intervals
**Pattern:** Greedy — Sort by End Time

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Sort by END time and greedily
keep the interval that finishes earliest — it leaves the
most room for what comes next. Every skip is one removal.
</div>

## Official Problem Statement

Given an array of intervals `intervals` where
`intervals[i] = [start_i, end_i]`, return the minimum
number of intervals you need to remove to make the
rest of the intervals non-overlapping.

**Example 1:**
```
Input:  intervals = [[1,2],[2,3],[3,4],[1,3]]
Output: 1
Explanation: Remove [1,3]; rest do not overlap.
```
**Example 2:**
```
Input:  intervals = [[1,2],[1,2],[1,2]]
Output: 2
Explanation: Remove two of the three [1,2].
```
**Example 3:**
```
Input:  intervals = [[1,2],[2,3]]
Output: 0
Explanation: Touching at boundary is not overlapping.
```

**Constraints:**
- `1 <= intervals.length <= 10^5`
- `-5 * 10^4 <= start_i < end_i <= 5 * 10^4`

## What This Is Actually Asking

You have calendar blocks, some overlapping.
Remove as few as possible so no two remaining
blocks collide.
Return only the count of removals — not which ones.
Touching at a boundary does not count as overlapping.

## Walk Through an Example by Hand

```
Input: [[1,2],[2,3],[3,4],[1,3]]

Sort by END:
  [1,2]  [2,3]  [1,3]  [3,4]

fence = -infinity   removals = 0

  [1,2]:  start=1 >= -inf  -> KEEP   fence=2
  [2,3]:  start=2 >= 2     -> KEEP   fence=3
  [1,3]:  start=1 <  3     -> REMOVE removals=1
  [3,4]:  start=3 >= 3     -> KEEP   fence=4

Answer: 1

Why [1,3] and not [2,3]?
  After sorting by end, [2,3] comes first (end=3 tie
  broken by start — [2,3] before [1,3]).
  We keep [2,3] because it was seen first in the
  sorted order. [1,3] arrives later and loses.
```

## The Picture

```
Sorted by end time:

  [1=2]             KEEP   fence=2
     [2===3]        KEEP   fence=3
  [1=====3]         REMOVE (starts before fence)
         [3====4]   KEEP   fence=4

Fence = end of the last kept interval.
Anything that starts before the fence -> remove.
Anything that starts at or after the fence -> keep.

  "fence" moves forward only when you KEEP.
  "removals" increments only when you SKIP.
```

## When To Use This Pattern

- When you see **minimum removals** from intervals,
  think **greedy — sort by end, count overlaps**
- When you see **maximum non-overlapping count**,
  think **same approach** (removals = total - kept)
- When making a greedy choice, think **keep the
  interval ending soonest — clears the most future
  space**
- When intervals touch at a boundary `[a,b],[b,c]`,
  think **not overlapping** — only `start < fence`
  triggers a removal

## The Approach

Sort all intervals by end time.
Walk through them tracking a fence — the end time
of the last interval you chose to keep.
If the current interval's start is before the fence,
it overlaps: remove it (count goes up).
Otherwise keep it and move the fence to its end.

In [ ]:
from typing import List  # type hints for the solution

In [ ]:
def test_harness(func):
    tests = [
        # (intervals, expected_removals)
        ([[1,2],[2,3],[3,4],[1,3]],       1),
        ([[1,2],[1,2],[1,2]],             2),  # all identical
        ([[1,2],[2,3]],                   0),  # touch only
        ([[1,100],[11,22],[1,11],[2,12]], 2),
        ([[1,2]],                         0),  # single interval
        ([[1,2],[3,4],[5,6]],             0),  # no overlaps
        ([[1,5],[2,3],[3,4]],             1),  # one long overlaps two
        ([[-5,-1],[0,2],[1,3]],           1),  # negatives
        ([[0,1],[0,1],[0,1],[0,1]],       3),  # many identical
        ([[1,3],[2,4],[3,5],[4,6]],       1),  # chain with one extra
    ]

    passed = 0
    for i, (intervals, expected) in enumerate(tests):
        result = func([x[:] for x in intervals])
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"expected={expected} | got={result}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [ ]:
def eraseOverlapIntervals(
    intervals: List[List[int]]
) -> int:
    """
    Return minimum intervals to remove so none overlap.

    Sort by end time. Walk sorted list with a fence
    tracking the last kept interval's end. If current
    start < fence, it overlaps — remove it. Otherwise
    keep it and advance the fence.

    Time:  O(n log n) — sort dominates
    Space: O(1) — only fence and counter tracked
    """
    pass


# Quick debug — run this cell while building
print(eraseOverlapIntervals(
    [[1,2],[2,3],[3,4],[1,3]]))   # 1
print(eraseOverlapIntervals(
    [[1,2],[1,2],[1,2]]))         # 2
print(eraseOverlapIntervals(
    [[1,2],[2,3]]))               # 0
print(eraseOverlapIntervals(
    [[1,5],[2,3],[3,4]]))         # 1

In [ ]:
# Uncomment and run when solution is ready
# test_harness(eraseOverlapIntervals)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force — try all subsets | O(2^n) | O(n) |
| Greedy — sort by end, count overlaps | O(n log n) | O(1) |

Sorting by end (not start) is the key — it makes the
locally optimal choice (keep earliest-ending)
globally optimal.

## Real World Connection

At Citi, ML forecasting jobs run on a shared cluster
with a fixed number of compute slots per time window.
When the cluster is overbooked, the scheduler must
cancel the fewest jobs to eliminate conflicts — exactly
this problem.
Sorting by end and greedily keeping early-finishing
jobs maximises the number of Prophet model training
runs that complete per day.
The same greedy logic resolves Lambda concurrency
conflicts in the AWS ETL pipeline when too many
batch windows overlap during peak load.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra